In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
# matplotlib.use('Agg')
%matplotlib inline

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.meta.env_stock_trading.env_forex_price_trailing import ForexPriceTrailingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl.meta.data_processor import DataProcessor
from finrl.meta.data_processors.processor_yahoofinance import YahooFinanceProcessor
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline
from pprint import pprint
import sys
import itertools

/opt/anaconda3/envs/ofbot/lib/python3.11/site-packages/pyfolio/pos.py:26: UserWarning: Module "zipline.assets" not found; mutltipliers will not be applied to position notionals.
  warnings.warn(


In [3]:
from finrl import config
from finrl import config_tickers
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)
check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

In [4]:
TRAIN_START_DATE = '2010-01-01'
TRAIN_END_DATE = '2021-10-01'
TRADE_START_DATE = '2021-10-01'
TRADE_END_DATE = '2023-03-01'

In [5]:
#df = YahooDownloader(start_date = TRAIN_START_DATE,
#                     end_date = TRADE_END_DATE,
#                     ticker_list = config_tickers.DOW_30_TICKER).fetch_data()
yfp = YahooFinanceProcessor()
df = yfp.scrap_data(['AXP', 'AMGN', 'AAPL'], '2010-01-01', TRADE_END_DATE)
print(df)

Processing AXP (1/3)... 33.33% complete.
Processing AMGN (2/3)... 66.67% complete.
Processing AAPL (3/3)... 100.00% complete.
           date    open    high     low   close   adjcp     volume   tic   day
0    2010-01-04    7.62    7.66    7.59    7.64    6.43  493729600  AAPL     3
1    2010-01-04   56.63   57.87   56.56   57.72   39.91    5277400  AMGN     3
2    2010-01-04   40.81   41.10   40.39   40.92   32.72    6894300   AXP     3
3    2010-01-05    7.66    7.70    7.62    7.66    6.44  601904800  AAPL     4
4    2010-01-05   57.33   57.69   56.27   57.22   39.57    7882800  AMGN     4
...         ...     ...     ...     ...     ...     ...        ...   ...   ...
9928 2023-02-27  235.22  235.22  232.89  234.45  218.18    1723000  AMGN  4805
9929 2023-02-27  175.53  175.69  173.08  173.30  168.38    1909000   AXP  4805
9930 2023-02-28  147.05  149.08  146.83  147.41  145.75   50547000  AAPL  4806
9931 2023-02-28  233.63  234.47  231.60  231.66  215.59    2615700  AMGN  4806
9932 

In [6]:
print(config_tickers.DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [7]:
df.sort_values(['date','tic'],ignore_index=True).head()

,date,open,high,low,close,adjcp,volume,tic,day
0,2010-01-04,7.62,7.66,7.59,7.64,6.43,493729600,AAPL,3
1,2010-01-04,56.63,57.87,56.56,57.72,39.91,5277400,AMGN,3
2,2010-01-04,40.81,41.10,40.39,40.92,32.72,6894300,AXP,3
3,2010-01-05,7.66,7.70,7.62,7.66,6.44,601904800,AAPL,4
4,2010-01-05,57.33,57.69,56.27,57.22,39.57,7882800,AMGN,4


In [8]:
fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = INDICATORS,
                    use_vix=True,
                    use_turbulence=True,
                    user_defined_feature = False)

processed = fe.preprocess_data(df)

Successfully added technical indicators
YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (3310, 8)
Successfully added vix


In [9]:
list_ticker = processed["tic"].unique().tolist()
list_date = list(pd.date_range(processed['date'].min(),processed['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

processed_full = pd.DataFrame(combination,columns=["date","tic"]).assign(date=lambda df: pd.to_datetime(df.date)).merge(processed,on=["date","tic"],how="left")
processed_full = processed_full[processed_full['date'].isin(processed['date'])]
processed_full = processed_full.sort_values(['date','tic'])

processed_full = processed_full.fillna(0)

In [10]:
processed_full.sort_values(['date','tic'],ignore_index=True).head(10)

,date,tic,open,high,low,close,adjcp,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix
0,2010-01-04,AAPL,7.62,7.66,7.59,7.64,6.43,493729600.0,3.0,0.000000,7.678284,7.621716,100.000000,66.666667,100.000000,7.640000,7.640000,20.040001
1,2010-01-04,AMGN,56.63,57.87,56.56,57.72,39.91,5277400.0,3.0,0.000000,7.678284,7.621716,100.000000,66.666667,100.000000,57.720000,57.720000,20.040001
2,2010-01-04,AXP,40.81,41.10,40.39,40.92,32.72,6894300.0,3.0,0.000000,7.678284,7.621716,100.000000,66.666667,100.000000,40.920000,40.920000,20.040001
3,2010-01-05,AAPL,7.66,7.70,7.62,7.66,6.44,601904800.0,4.0,0.000449,7.678284,7.621716,100.000000,66.666667,100.000000,7.650000,7.650000,19.350000
4,2010-01-05,AMGN,57.33,57.69,56.27,57.22,39.57,7882800.0,4.0,-0.011218,58.177107,56.762893,0.000000,-66.666667,100.000000,57.470000,57.470000,19.350000
5,2010-01-05,AXP,40.83,41.23,40.37,40.83,32.65,10641200.0,4.0,-0.002019,41.002279,40.747721,0.000000,66.666667,100.000000,40.875000,40.875000,19.350000
6,2010-01-06,AAPL,7.66,7.69,7.53,7.53,6.34,552160000.0,5.0,-0.003460,7.750000,7.470000,12.946429,-100.000000,39.896373,7.610000,7.610000,19.160000
7,2010-01-06,AMGN,56.94,57.39,56.50,56.79,39.27,6015100.0,5.0,-0.027628,58.174211,56.312456,0.000000,-80.737705,100.000000,57.243333,57.243333,19.160000
8,2010-01-06,AXP,41.23,41.67,41.17,41.49,33.18,8399400.0,5.0,0.017894,41.795821,40.364179,88.353414,100.000000,100.000000,41.080000,41.080000,19.160000
9,2010-01-07,AAPL,7.56,7.57,7.47,7.52,6.33,477131200.0,6.0,-0.005513,7.732988,7.442012,12.107688,-111.904762,59.455225,7.587500,7.587500,19.059999


In [11]:
mvo_df = processed_full.sort_values(['date','tic'],ignore_index=True)[['date','tic','close']]

In [12]:
train = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE)
trade = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE)
train_length = len(train)
trade_length = len(trade)
print(train_length)
print(trade_length)

8871
1059


In [13]:
stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 3, State Space: 31


In [14]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimension
num_stock_shares = [0] * stock_dimension

env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4
}


e_train_gym = StockTradingEnv(df = train, **env_kwargs)

In [15]:
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [16]:
agent = DRLAgent(env = env_train)

if_using_a2c = True
if_using_ddpg = True
if_using_ppo = True
if_using_td3 = True
if_using_sac = True


In [17]:
agent = DRLAgent(env = env_train)
model_a2c = agent.get_model("a2c")

if if_using_a2c:
  # set up logger
  tmp_path = RESULTS_DIR + '/a2c'
  new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
  # Set new logger
  model_a2c.set_logger(new_logger_a2c)


{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c


In [18]:
trained_a2c = agent.train_model(model=model_a2c, 
                             tb_log_name='a2c',
                             total_timesteps=50000) if if_using_a2c else None

--------------------------------------
| time/                 |            |
|    fps                | 0          |
|    iterations         | 100        |
|    time_elapsed       | 2647       |
|    total_timesteps    | 500        |
| train/                |            |
|    entropy_loss       | -4.31      |
|    explained_variance | -0.308     |
|    learning_rate      | 0.0007     |
|    n_updates          | 99         |
|    policy_loss        | 0.76       |
|    reward             | 0.42689    |
|    reward_max         | 3.2076826  |
|    reward_mean        | 1.1716273  |
|    reward_min         | 0.38081327 |
|    std                | 1.02       |
|    value_loss         | 0.53       |
--------------------------------------
--------------------------------------
| time/                 |            |
|    fps                | 0          |
|    iterations         | 200        |
|    time_elapsed       | 2648       |
|    total_timesteps    | 1000       |
| train/                |

In [18]:
data_risk_indicator = processed_full[(processed_full.date<TRAIN_END_DATE) & (processed_full.date>=TRAIN_START_DATE)]
insample_risk_indicator = data_risk_indicator.drop_duplicates(subset=['date'])

In [19]:
insample_risk_indicator.vix.describe()

count    2957.000000
mean       18.105293
std         7.272476
min         9.140000
25%        13.370000
50%        16.209999
75%        20.629999
max        82.690002
Name: vix, dtype: float64

In [20]:
insample_risk_indicator.vix.quantile(0.996)

np.float64(57.212001831054636)

In [21]:
e_trade_gym = StockTradingEnv(df = trade, turbulence_threshold = None,risk_indicator_col='vix', **env_kwargs)
# env_trade, obs_trade = e_trade_gym.get_sb_env()

In [22]:
trade.head()

,date,tic,open,high,low,close,adjcp,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix
0,2021-10-01,AAPL,141.90,142.92,139.11,142.65,139.81,94639600.0,4291.0,-1.728208,156.751718,138.340282,46.861483,-142.138308,21.509145,148.468333,147.631000,21.100000
0,2021-10-01,AMGN,213.59,214.61,210.80,213.92,189.70,2629400.0,4291.0,-3.377972,223.283034,209.232966,39.754599,-96.444581,30.123478,218.784000,228.629667,21.100000
0,2021-10-01,AXP,168.50,175.12,168.48,173.94,165.85,3956000.0,4291.0,2.322256,178.188111,152.632889,56.228932,117.704163,12.003926,164.888667,167.183000,21.100000
1,2021-10-04,AAPL,141.76,142.21,138.27,139.14,136.37,98322000.0,4294.0,-2.046247,156.147408,137.428592,43.304728,-158.854221,23.278106,148.166667,147.531500,22.959999
1,2021-10-04,AMGN,214.10,215.64,210.77,211.44,187.50,2856300.0,4294.0,-3.395664,221.021931,210.001069,38.055501,-99.081401,27.176141,218.381000,228.067000,22.959999


In [23]:
trained_moedl = trained_a2c
df_account_value_a2c, df_actions_a2c = DRLAgent.DRL_prediction(
    model=trained_moedl, 
    environment = e_trade_gym)

hit end!


In [24]:
df_account_value_a2c.shape

(353, 2)

In [25]:
def process_df_for_mvo(df):
  df = df.sort_values(['date','tic'],ignore_index=True)[['date','tic','close']]
  fst = df
  fst = fst.iloc[0:stock_dimension, :]
  tic = fst['tic'].tolist()

  mvo = pd.DataFrame()

  for k in range(len(tic)):
    mvo[tic[k]] = 0

  for i in range(df.shape[0]//stock_dimension):
    n = df
    n = n.iloc[i * stock_dimension:(i+1) * stock_dimension, :]
    date = n['date'][i*stock_dimension]
    mvo.loc[date] = n['close'].tolist()
  
  return mvo

In [26]:
# Codes in this section partially refer to Dr G A Vijayalakshmi Pai

# https://www.kaggle.com/code/vijipai/lesson-5-mean-variance-optimization-of-portfolios/notebook

def StockReturnsComputing(StockPrice, Rows, Columns): 
  import numpy as np 
  StockReturn = np.zeros([Rows-1, Columns]) 
  for j in range(Columns):        # j: Assets 
    for i in range(Rows-1):     # i: Daily Prices 
      StockReturn[i,j]=((StockPrice[i+1, j]-StockPrice[i,j])/StockPrice[i,j])* 100 
      
  return StockReturn

In [27]:
train_mvo = data_split(processed_full, TRAIN_START_DATE,TRAIN_END_DATE).reset_index()
trade_mvo = data_split(processed_full, TRADE_START_DATE,TRADE_END_DATE).reset_index()

In [28]:
StockData = process_df_for_mvo(train_mvo)
TradeData = process_df_for_mvo(trade_mvo)

TradeData.to_numpy()

array([[142.65, 213.92, 173.94],
       [139.14, 211.44, 172.66],
       [141.11, 211.86, 174.76],
       ...,
       [149.4 , 237.62, 175.14],
       [146.71, 233.66, 174.25],
       [147.92, 234.45, 173.3 ]], shape=(353, 3))

In [29]:
#compute asset returns
arStockPrices = np.asarray(StockData)
[Rows, Cols]=arStockPrices.shape
arReturns = StockReturnsComputing(arStockPrices, Rows, Cols)

#compute mean returns and variance covariance matrix of returns
meanReturns = np.mean(arReturns, axis = 0)
covReturns = np.cov(arReturns, rowvar=False)
 
#set precision for printing results
np.set_printoptions(precision=3, suppress = True)

#display mean returns and variance-covariance matrix of returns
print('Mean returns of assets in k-portfolio 1\n', meanReturns)
print('Variance-Covariance matrix of returns\n', covReturns)

Mean returns of assets in k-portfolio 1
 [0.115 0.056 0.064]
Variance-Covariance matrix of returns
 [[3.146 1.018 1.291]
 [1.018 2.407 1.073]
 [1.291 1.073 3.305]]


In [30]:
from pypfopt.efficient_frontier import EfficientFrontier

ef_mean = EfficientFrontier(meanReturns, covReturns, weight_bounds=(0, 0.5))
raw_weights_mean = ef_mean.max_sharpe()
cleaned_weights_mean = ef_mean.clean_weights()
mvo_weights = np.array([1000000 * cleaned_weights_mean[i] for i in range(3)])
mvo_weights

array([500000., 304450., 195550.])

In [31]:
LastPrice = np.array([1/p for p in StockData.tail(1).to_numpy()[0]])
Initial_Portfolio = np.multiply(mvo_weights, LastPrice)
Initial_Portfolio

array([3533.569, 1431.695, 1167.254])

In [32]:
Portfolio_Assets = TradeData @ Initial_Portfolio
MVO_result = pd.DataFrame(Portfolio_Assets, columns=["Mean Var"])
# MVO_result

In [33]:
df_result_a2c = df_account_value_a2c.set_index(df_account_value_a2c.columns[0])
df_result_a2c.rename(columns = {'account_value':'a2c'}, inplace = True)
# df_result_ddpg = df_account_value_ddpg.set_index(df_account_value_ddpg.columns[0])
# df_result_ddpg.rename(columns = {'account_value':'ddpg'}, inplace = True)
# df_result_td3 = df_account_value_td3.set_index(df_account_value_td3.columns[0])
# df_result_td3.rename(columns = {'account_value':'td3'}, inplace = True)
# df_result_ppo = df_account_value_ppo.set_index(df_account_value_ppo.columns[0])
# df_result_ppo.rename(columns = {'account_value':'ppo'}, inplace = True)
# df_result_sac = df_account_value_sac.set_index(df_account_value_sac.columns[0])
# df_result_sac.rename(columns = {'account_value':'sac'}, inplace = True)
df_account_value_a2c.to_csv("df_account_value_a2c.csv")
#baseline stats
print("==============Get Baseline Stats===========")
df_dji_ = get_baseline(
        ticker="^DJI", 
        start = TRADE_START_DATE,
        end = TRADE_END_DATE)
stats = backtest_stats(df_dji_, value_col_name = 'close')
df_dji = pd.DataFrame()
df_dji['date'] = df_account_value_a2c['date']
df_dji['account_value'] = df_dji_['close'] / df_dji_['close'][0] * env_kwargs["initial_amount"]
df_dji.to_csv("df_dji.csv")
df_dji = df_dji.set_index(df_dji.columns[0])
df_dji.to_csv("df_dji+.csv")

result = pd.DataFrame()
result = pd.merge(result, df_result_a2c, how='outer', left_index=True, right_index=True)
# result = pd.merge(result, df_result_ddpg, how='outer', left_index=True, right_index=True)
# result = pd.merge(result, df_result_td3, how='outer', left_index=True, right_index=True)
# result = pd.merge(result, df_result_ppo, how='outer', left_index=True, right_index=True)
# result = pd.merge(result, df_result_sac, how='outer', left_index=True, right_index=True)
result = pd.merge(result, MVO_result, how='outer', left_index=True, right_index=True)
print(result.head())
result = pd.merge(result, df_dji, how='outer', left_index=True, right_index=True)
# result.columns = ['a2c', 'ddpg', 'td3', 'ppo', 'sac', 'mean var', 'dji']

# print("result: ", result)
result.to_csv("result.csv")

==============Get Baseline Stats===========


[*********************100%***********************]  1 of 1 completed

Shape of DataFrame:  (354, 8)
Annual return         -0.034876
Cumulative returns    -0.048644
Annual volatility      0.181612
Sharpe ratio          -0.105351
Calmar ratio          -0.158953
Stability              0.280983
Max drawdown          -0.219408
Omega ratio            0.982546
Sortino ratio         -0.146974
Skew                        NaN
Kurtosis                    NaN
Tail ratio             0.970602
Daily value at risk   -0.022957
dtype: float64
                    a2c      Mean Var
date                                 
2021-10-01  1000000.000  1.013364e+06
2021-10-04   999219.949  9.959164e+05
2021-10-05  1000065.625  1.005930e+06
2021-10-06   999307.852  1.004796e+06
2021-10-07   999783.314  1.009162e+06
